In [ ]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
def t_test(non_mace_avg, non_mace_std, non_mace_group_size, mace_avg, mace_std, mace_group_size): # std 樣本標準差

    t_statistic, p_value = stats.ttest_ind_from_stats(mean1=non_mace_avg, std1=non_mace_std, nobs1=non_mace_group_size, mean2=mace_avg, std2=mace_std, nobs2=mace_group_size)
    
    return t_statistic, p_value

In [ ]:
data_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_rpeak'
save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_rpeak'

In [ ]:
non_mace_data = pd.read_csv(os.path.join(data_path, 'non MACE_rpeak_counts_metric.csv'))
mace_data = pd.read_csv(os.path.join(data_path, 'MACE_rpeak_counts_metric.csv'))

In [ ]:
non_mace_rpeaks_data = non_mace_data.iloc[:, 1:].values.astype(np.float32)
mace_rpeaks_data = mace_data.iloc[:, 1:].values.astype(np.float32)

In [ ]:
print(non_mace_rpeaks_data.shape, mace_rpeaks_data.shape)

In [ ]:
row_names = [
    'rpeaks_counts_fc500',
    'rpeaks_counts_fc150',
    'rpeaks_counts_fc40',
    'r40-r500',
    'r40-r150',
    'r150-r500',
]

In [ ]:
tmp_dict = {"approach": row_names, "Non MACE_avg": [], "Non MACE_std": [], "MACE_avg": [], "MACE_std": [], 'p_value': [], 't_test': []}

for i in tqdm(range(non_mace_rpeaks_data.shape[1])):
    non_mace_avg = np.mean(non_mace_rpeaks_data[:, i])
    non_mace_std = np.std(non_mace_rpeaks_data[:, i], ddof=1)
    mace_avg = np.mean(mace_rpeaks_data[:, i])
    mace_std = np.std(mace_rpeaks_data[:, i], ddof=1)
    t_statistic, p_value = t_test(non_mace_avg, non_mace_std, non_mace_rpeaks_data.shape[0], mace_avg, mace_std, mace_rpeaks_data.shape[0])
    
    tmp_dict['Non MACE_avg'].append(non_mace_avg)
    tmp_dict['Non MACE_std'].append(non_mace_std)
    tmp_dict['MACE_avg'].append(mace_avg)
    tmp_dict['MACE_std'].append(mace_std)
    tmp_dict['p_value'].append(p_value)
    tmp_dict['t_test'].append(t_statistic)

rpeak_statis_df = pd.DataFrame(tmp_dict)
display(rpeak_statis_df)
rpeak_statis_df.to_csv(os.path.join(save_path, 'rpeak_static_metric.csv'), index=False)

In [ ]:
x_index = np.arange(len(rpeak_statis_df['approach']))
bar_width = 0.4
plt.figure(figsize=(14, 7))
plt.bar(x_index-bar_width, rpeak_statis_df['Non MACE_avg'], label='Non MACE avg', width=bar_width, alpha=0.6, color='blue', align='edge')
plt.errorbar(x_index-bar_width/2, rpeak_statis_df['Non MACE_avg'], yerr=rpeak_statis_df['Non MACE_std'], fmt='o', color='blue', capsize=10, label='Non MACE std')
plt.bar(x_index, rpeak_statis_df['MACE_avg'], width=bar_width, label='MACE avg', alpha=0.6, color='red', align='edge')
plt.errorbar(x_index+bar_width/2, rpeak_statis_df['MACE_avg'], yerr=rpeak_statis_df['MACE_std'], fmt='o', color='red', capsize=10, label='MACE std')

plt.xticks(x_index, rpeak_statis_df['approach'])

for i in range(len(rpeak_statis_df['approach'])):
    plt.text(x_index[i]-bar_width, rpeak_statis_df['Non MACE_avg'][i] + 5, f'{rpeak_statis_df["Non MACE_avg"][i]:.3f}', color='blue', fontsize=10, ha='center')
    plt.text(x_index[i]-bar_width/2, rpeak_statis_df['Non MACE_avg'][i] + rpeak_statis_df['Non MACE_std'][i] + 5, f'{rpeak_statis_df["Non MACE_std"][i]:.3f}', color='blue', fontsize=10, ha='center')
    plt.text(x_index[i], rpeak_statis_df['MACE_avg'][i] + 5, f'{rpeak_statis_df["MACE_avg"][i]:.3f}', color='red', fontsize=10, ha='center')
    plt.text(x_index[i]+bar_width/2, rpeak_statis_df['MACE_avg'][i] + rpeak_statis_df['MACE_std'][i] + 5, f'{rpeak_statis_df["MACE_std"][i]:.3f}', color='red', fontsize=10, ha='center')

plt.legend()
plt.title('Comparison of Metrics: Non MACE vs. MACE', fontsize=14, fontweight='bold')
plt.ylabel('Average Value')
plt.xlabel('Approach')
plt.tight_layout()
plt.grid(axis='y', linestyle='-', alpha=0.7)
plt.savefig(os.path.join(save_path, 'rpeak_static_metric.png'), dpi=600, bbox_inches='tight')
plt.show()